# Predicción del test: multiclass (0 = mel, 1 = bcc, 2 = scc)

Carga un modelo ya entrenado (de la carpeta de resultados de uno de los notebooks `03_v*`), predice el test y guarda el
CSV de la entrega: **una sola columna, 0 = mel, 1 = bcc y 2 = scc, en el orden de la numeración de las imágenes**.

**Ojo con la numeración:** en los notebooks de entrenamiento las clases son `0 = bcc, 1 = mel, 2 = scc` (orden
alfabético), pero la entrega pide `0 = mel, 1 = bcc, 2 = scc`. Este notebook lo convierte (`SUBMIT_CODES`) y lo comprueba
en val.

**Lo único que hay que tocar es la primera celda:**
- `RESULTS_DIR`: la carpeta de resultados del notebook de entrenamiento:

  | notebook | carpeta |
  |---|---|
  | v1 (pesos de clase) | `ml/ml_results_multiclass` |
  | v2 (ROS) | `ml/ml_results_multiclass_ros` |
  | v3 (RUS) | `ml/ml_results_multiclass_rus` |
  | v4 (SMOTE) | `ml/ml_results_multiclass_smote` |
  | v5 (RUS+SMOTE middle) | `ml/ml_results_multiclass_rus_smote` |
  | v6 (RUS+SMOTE 1500) | `ml/ml_results_multiclass_rus_smote_1500` |
  | v7 (PCA/LDA, RUS+SMOTE) | `ml/ml_results_multiclass_pca_lda` |
  | v8 (PCA/LDA, SMOTE) | `ml/ml_results_multiclass_smote_pca_lda` |

- `MODEL_FILE`: qué modelo de `RESULTS_DIR/models/` usar. La segunda celda los lista todos con su nombre:
  - `final_model_trainval.joblib`: el elegido, reentrenado con train + val (sección 6 del notebook de entrenamiento).
  - `best_model.joblib`: el mejor de la CV, entrenado solo con train.
  - `<modelo>.joblib`: cualquier otro (`ens_hgb_svm.joblib`, `svm_rbf.joblib`...), entrenado solo con train.
- `WEIGHTS`: factores de decisión por clase (4c). `"auto"` usa los guardados para ese modelo (si no hay, ninguno:
  argmax normal). También se pueden poner a mano, p. ej. `{"bcc": 1.0, "mel": 1.0, "scc": 1.5}`, o `None` para argmax.

Antes de predecir el test, comprueba el mismo modelo con los mismos factores **en val**: si el kappa cuadra con el del
notebook de entrenamiento, se ha cargado lo que toca.

Necesita `features_test_multiclass_final/features_test.csv` (notebooks 05 y 06 de Multiclass) y las imágenes de
`test_multiclass/` (para comprobar que hay una predicción por imagen y ordenarlas).

In [1]:
import json
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, confusion_matrix, recall_score

warnings.filterwarnings("ignore")

#ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Multiclass").is_dir()), Path.cwd())
ROOT = Path(r"C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG")    # o fíjala a mano

# ======================= LO QUE SE CAMBIA =======================
RESULTS_DIR = ROOT / "ml" / "ml_results_multiclass_smote"    # carpeta de resultados del notebook de entrenamiento (tabla de arriba)
MODEL_FILE = "ens_hgb_svm.joblib"                    # archivo de RESULTS_DIR/models/ (la celda siguiente los lista)
WEIGHTS = "auto"                                              # "auto" | None (argmax) | {"bcc": 1.0, "mel": 1.0, "scc": 1.5}
# ================================================================

TEST_CSV = ROOT / "features_test_multiclass_final" / "features_test.csv"
TEST_IMG_DIR = ROOT / "test_multiclass"             # imágenes originales del test (para no perder ninguna y ordenarlas)
OUT_CSV = ROOT / "submission" / "test_multiclass_predictions.csv"
CHECK_ON_VAL = True                                  # comprueba el modelo + factores en val antes de predecir el test
SUBMIT_CODES = {"mel": 0, "bcc": 1, "scc": 2}       # códigos de la ENTREGA (distintos de los del entrenamiento)
SUBMIT_NAMES = {v: k for k, v in SUBMIT_CODES.items()}

MODEL_DIR = RESULTS_DIR / "models"
assert MODEL_DIR.is_dir(), f"No existe {MODEL_DIR}: ¿has ejecutado el notebook de entrenamiento con esa carpeta?"
print("ROOT:", ROOT)

ROOT: C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG


## 1. Modelos disponibles

In [2]:
def read_json(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


info_all = read_json(MODEL_DIR / "models_info.json")              # modelos entrenados con train ("Guardar los modelos")
info_final = read_json(MODEL_DIR / "final_model_trainval.json")   # modelo reentrenado con train + val (última sección)

file2name = {f: n for n, f in info_all.get("files", {}).items() if n != "best"}
if info_all.get("best_model"):
    file2name["best_model.joblib"] = f"{info_all['best_model']}  (best_model: el mejor de la CV)"
if info_final:
    file2name["final_model_trainval.joblib"] = f"{info_final['model']}  (reentrenado con train + val)"

print(f"Modelos disponibles en {MODEL_DIR} (pon el archivo en MODEL_FILE):\n")
for f in sorted(MODEL_DIR.glob("*.joblib")):
    mark = "  <-- elegido" if f.name == MODEL_FILE else ""
    print(f"  {f.name:<38s} {file2name.get(f.name, ''):<55s} {f.stat().st_size / 1e6:6.1f} MB{mark}")
assert (MODEL_DIR / MODEL_FILE).exists(), f"No existe {MODEL_DIR / MODEL_FILE}: elige uno de la lista"

Modelos disponibles en C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\ml\ml_results_multiclass_smote\models (pon el archivo en MODEL_FILE):

  best_model.joblib                      Ens HGB+SVM+LogReg  (best_model: el mejor de la CV)       12.4 MB
  decision_tree.joblib                   Decision Tree                                              0.5 MB
  ens_hgb_svm.joblib                     Ens HGB+SVM                                               12.4 MB  <-- elegido
  ens_hgb_svm__svm_x2.joblib             Ens HGB+SVM (SVM x2)                                      12.4 MB
  ens_hgb_svm_logreg.joblib              Ens HGB+SVM+LogReg                                        12.4 MB
  ens_xgb_hgb_svm.joblib                 Ens XGB+HGB+SVM                                           13.7 MB
  ensemble__xgb_hgb_svm.joblib                                                                     15.6 MB
  final_model_trainval.joblib            Ens HGB+SVM+LogReg  (reentrenado con train + v

## 2. Cargar el modelo, las clases y los factores de decisión

In [3]:
model = joblib.load(MODEL_DIR / MODEL_FILE)
is_final = MODEL_FILE == "final_model_trainval.joblib"
if is_final:
    MODEL_NAME = info_final["model"]
elif MODEL_FILE == "best_model.joblib":
    MODEL_NAME = info_all.get("best_model", MODEL_FILE)
else:
    MODEL_NAME = file2name.get(MODEL_FILE, MODEL_FILE)
TRAINED_ON = "train + val" if is_final else "train"
info = info_final if is_final else info_all
FEATS = info.get("features") or info_all["features"]
labels = info.get("labels") or info_all["labels"]                       # {"0": "bcc", "1": "mel", "2": "scc"}
CLASSES = [labels[str(i)] for i in range(len(labels))]                  # orden interno del modelo
TO_SUBMIT = np.array([SUBMIT_CODES[c] for c in CLASSES])                # índice interno -> código de la entrega


def proba(X):
    """Probabilidades (n, 3) en el orden de CLASSES. Si el modelo no tiene predict_proba (SVM), softmax de decision_function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    d = model.decision_function(X)
    e = np.exp(d - d.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


w_saved = read_json(RESULTS_DIR / "decision_weights.json")
if WEIGHTS == "auto":
    if is_final and "decision_weights" in info_final:
        w, w_src = np.array(info_final["decision_weights"], float), "final_model_trainval.json"
    elif w_saved.get("model") == MODEL_NAME:
        w, w_src = np.array(w_saved["weights"], float), "decision_weights.json"
    else:
        w, w_src = np.ones(len(CLASSES)), "ninguno (no hay factores guardados para este modelo): argmax"
elif WEIGHTS is None:
    w, w_src = np.ones(len(CLASSES)), "ninguno: argmax"
else:
    w, w_src = np.array([float(WEIGHTS.get(c, 1.0)) for c in CLASSES]), "fijados a mano"
DECISION_INFO = {"weights": dict(zip(CLASSES, w.round(3).tolist())), "source": w_src}


def predict_internal(X):
    return np.argmax(proba(X) * w, axis=1)


print(f"Modelo:     {MODEL_NAME}  [{MODEL_FILE}]")
print(f"Entrenado:  con {TRAINED_ON} | {len(FEATS)} features")
print(f"Clases del modelo:  {dict(enumerate(CLASSES))}")
print(f"Clases de la entrega: {SUBMIT_NAMES}")
print(f"Factores:   {DECISION_INFO['weights']}  ({w_src})")

Modelo:     Ens HGB+SVM  [ens_hgb_svm.joblib]
Entrenado:  con train | 412 features
Clases del modelo:  {0: 'bcc', 1: 'mel', 2: 'scc'}
Clases de la entrega: {0: 'mel', 1: 'bcc', 2: 'scc'}
Factores:   {'bcc': 1.0, 'mel': 1.0, 'scc': 1.0}  (ninguno (no hay factores guardados para este modelo): argmax)


## 3. Comprobación en val (y de la conversión de etiquetas)
Mismo modelo y mismos factores sobre val. Si el modelo se entrenó solo con train, el kappa debe coincidir con el del
notebook de entrenamiento (con factores, con el de la sección 4c). Si es el reentrenado con train + val, saldrá más alto
de lo real: solo comprueba que todo cuadra. También se comprueba que, tras convertir a los códigos de la entrega, la
matriz de confusión sigue siendo la misma (MEL con MEL, BCC con BCC...).

In [4]:
if CHECK_ON_VAL:
    cand = []
    if info_all.get("train_csv"):
        cand.append(Path(str(info_all["train_csv"]).replace("features_train", "features_val")))
    cand += [ROOT / d / "features_val.csv" for d in ("features_multiclass", "features_multiclass_final")]
    val_csv = next((p for p in cand if p.exists()), None)
    if val_csv is None:
        print("No encuentro el features_val.csv de multiclass: se omite la comprobación")
    else:
        va = pd.read_csv(val_csv)
        va = va[va["subtype"].isin(CLASSES)].reset_index(drop=True)
        yv = va["subtype"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
        pv = predict_internal(va[FEATS])
        rec = recall_score(yv, pv, labels=list(range(len(CLASSES))), average=None)
        print(f"Val ({len(yv)} imágenes, {val_csv.parent.name}/): kappa {cohen_kappa_score(yv, pv):.4f} | "
              f"bal_acc {balanced_accuracy_score(yv, pv):.4f} | recall " +
              ", ".join(f"{c} {r:.3f}" for c, r in zip(CLASSES, rec)))

        # misma matriz con los códigos de la entrega: la diagonal debe ser la misma
        ys, ps = TO_SUBMIT[yv], TO_SUBMIT[pv]
        order = sorted(SUBMIT_NAMES)
        cm = pd.DataFrame(confusion_matrix(ys, ps, labels=order),
                          index=[f"real {k}={SUBMIT_NAMES[k]}" for k in order],
                          columns=[f"pred {k}={SUBMIT_NAMES[k]}" for k in order])
        print("\nMatriz de confusión con los códigos de la ENTREGA:\n" + cm.to_string())
        assert np.isclose(cohen_kappa_score(ys, ps), cohen_kappa_score(yv, pv)), "la conversión de etiquetas no cuadra"
        if is_final:
            print("\nOJO: este modelo se entrenó también con val -> la cifra sale optimista (solo comprueba que todo cuadra).")

Val (1270 imágenes, features_multiclass/): kappa 0.7976 | bal_acc 0.7543 | recall bcc 0.934, mel 0.925, scc 0.404

Matriz de confusión con los códigos de la ENTREGA:
            pred 0=mel  pred 1=bcc  pred 2=scc
real 0=mel         627          49           2
real 1=bcc          26         465           7
real 2=scc          20          36          38


## 4. Predicción del test (una fila por imagen, en orden, con los códigos de la entrega)

In [5]:
def image_number(name):
    nums = re.findall(r"\d+", str(name))
    return int(nums[-1]) if nums else None


te = pd.read_csv(TEST_CSV)
missing_cols = [c for c in FEATS if c not in te.columns]
assert not missing_cols, f"Al CSV del test le faltan {len(missing_cols)} features (p. ej. {missing_cols[:5]})"
te["name"] = te["name"].astype(str)
assert te["name"].is_unique, "hay nombres repetidos en el CSV del test"
n_nan = int(te[FEATS].isna().any(axis=1).sum())
print(f"Test: {len(te)} imágenes con features | {n_nan} con algún NaN (se imputan con la mediana de train, como en val)")

# lista completa de imágenes del test, para que no falte ninguna fila
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
if TEST_IMG_DIR.is_dir():
    all_names = [f.stem for f in TEST_IMG_DIR.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT]
    print(f"Imágenes en {TEST_IMG_DIR.name}/: {len(all_names)}")
else:
    print(f"AVISO: no encuentro {TEST_IMG_DIR}: se usan solo las imágenes del CSV (no se puede comprobar si falta alguna)")
    all_names = list(te["name"])

# orden = numeración de los archivos de imagen
nums = [image_number(n) for n in all_names]
if any(n is None for n in nums) or len(set(nums)) != len(nums):
    print("AVISO: no todos los nombres tienen un número único -> se ordenan alfabéticamente")
    ORDER = sorted(all_names)
else:
    ORDER = [n for _, n in sorted(zip(nums, all_names))]

pred_by_name = pd.Series(TO_SUBMIT[predict_internal(te[FEATS])], index=te["name"])     # ya en códigos de la entrega

missing = [n for n in ORDER if n not in pred_by_name.index]
extra = sorted(set(pred_by_name.index) - set(ORDER))
if missing:
    fallback = int(pred_by_name.mode()[0])
    print(f"AVISO: {len(missing)} imágenes del test NO tienen features (falló el preprocesado o la extracción): "
          f"{missing[:10]}\n       -> se les asigna la clase más frecuente ({fallback}). Mejor revisarlas y extraerlas.")
    pred_by_name = pd.concat([pred_by_name, pd.Series(fallback, index=missing)])
if extra:
    print(f"AVISO: {len(extra)} nombres del CSV no están entre las imágenes del test (no se entregan): {extra[:10]}")

pred = pred_by_name.reindex(ORDER).astype(int)
print(f"\n{len(pred)} predicciones, en orden | primeras: {list(pred.index[:3])} ... última: {pred.index[-1]}")
print("recuento:", {f"{k} ({SUBMIT_NAMES[k]})": int((pred == k).sum()) for k in sorted(SUBMIT_NAMES)})

Test: 2121 imágenes con features | 0 con algún NaN (se imputan con la mediana de train, como en val)
AVISO: no encuentro C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\test_multiclass: se usan solo las imágenes del CSV (no se puede comprobar si falta alguna)

2121 predicciones, en orden | primeras: ['xxx00001', 'xxx00002', 'xxx00003'] ... última: xxx02121
recuento: {'0 (mel)': 1137, '1 (bcc)': 918, '2 (scc)': 66}


## 5. Guardar el CSV de la entrega
- `submission/test_multiclass_predictions.csv`: **lo que se entrega** (una columna, 0 = mel, 1 = bcc, 2 = scc).
- `..._con_nombres.csv`: lo mismo con el nombre de cada imagen y la clase en texto, para revisar (no se entrega).
- `.json`: qué modelo, factores y carpeta se usaron.

Si el profesor quiere cabecera, cambia `header=False` por `header=["label"]`.

In [6]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pred.to_csv(OUT_CSV, index=False, header=False)                     # LO QUE SE ENTREGA: una sola columna, sin cabecera
check_csv = OUT_CSV.with_name(OUT_CSV.stem + "_con_nombres.csv")    # solo para revisar (no se entrega)
pred.rename("pred").rename_axis("name").to_frame().assign(clase=pred.map(SUBMIT_NAMES).values).to_csv(check_csv)
OUT_CSV.with_suffix(".json").write_text(json.dumps(
    {"model": MODEL_NAME, "model_file": MODEL_FILE, "results_dir": str(RESULTS_DIR), "trained_on": TRAINED_ON,
     "decision": DECISION_INFO, "codes": SUBMIT_NAMES, "n_images": len(pred)},
    indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("PARA ENTREGAR:", OUT_CSV)
print("para revisar: ", check_csv, "\n")
print(pd.read_csv(OUT_CSV, header=None).head(5).to_string(header=False))

PARA ENTREGAR: C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\submission\test_multiclass_predictions.csv
para revisar:  C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG\submission\test_multiclass_predictions_con_nombres.csv 

0  1
1  1
2  1
3  0
4  0
